# Quickstart 4 — interpretation: from amplitudes to a companion mass

Everything measured is an amplitude, a period and a shape. A mass needs one
assumption on top: the primary's mass, and — for the RV channel — the
inclination. This notebook walks the observables-first ladder and names each
assumption where it enters: mass functions, the companion-mass solve, the
astrometric upper limit, the RV ↔ astrometry bridges, the flux-ratio and AMRF
triage, posterior summaries, and the prior family you can call yourself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from orblet.constants import DAYS_PER_KEPLER_YEAR
from orblet.simulate.bundles import load_simulated_inputs
from orblet import (prepare_rv_for_orbit, resolve_epochs_mjd, best_linear_params_rv,
                            best_linear_params_ti, recover_K, ti_amplitude_chains, ti_to_kepler)

bundle = load_simulated_inputs(seed=1)
truth = bundle.truth
astro = bundle.astro_data
prepared = prepare_rv_for_orbit(bundle.rv_data, time_scale="gaia_obmt")
EPOCH_REF = float(truth.t_ref_mjd)
TAU = ((truth.tp_mjd - EPOCH_REF) / truth.P_days) % 1.0
t_mjd = resolve_epochs_mjd(astro)

# Amplitude draws at the true shape from the linear solves (quickstart 2).
blp_rv = best_linear_params_rv(truth.P_days, truth.e, TAU, epochs_mjd=prepared["epochs_mjd"],
                               rv=prepared["rv"], rv_err=prepared["rv_err"], epoch_ref_mjd=EPOCH_REF, draw=2000)
K_draws = np.array([recover_K(b) for b in blp_rv.draws])
blp_ti = best_linear_params_ti(truth.P_days, truth.e, TAU, epochs_mjd=t_mjd,
                               scan_angle=np.asarray(astro["scan_angle"], float),
                               parallax_factor_al=np.asarray(astro["parallax_factor_al"], float),
                               centroid_pos=np.asarray(astro["centroid_pos"], float),
                               centroid_pos_err=np.asarray(astro["centroid_pos_err"], float),
                               epoch_ref_mjd=EPOCH_REF, draw=2000)
ti_chains = ti_to_kepler(ti_amplitude_chains(blp_ti.draws))
n = K_draws.size
P_days = np.full(n, truth.P_days); e = np.full(n, truth.e)
print(f"K = {np.median(K_draws):.2f} km/s; a_phot = {np.median(ti_chains['a_phot_mas']):.3f} mas; "
      f"plx = {np.median(ti_chains['plx_mas']):.3f} mas; truth m1 = {truth.m1_msun:.2f}, m2 = {truth.m2_msun:.2f} Msun")

## Mass functions: what each channel measures

The RV channel measures a₁ sin i (`a1sini_au`) and hence the spectroscopic mass
function (m₂ sin i)³/M² (`mass_function_au`). Astrometry measures the
photocentre axis (`a_phot_au`) and hence m₂³/M² — **if** the companion is dark
(β = 0, so a_phot = a₁). Neither is a mass yet.

In [ ]:
from orblet.interpret.astrometric_upper_limit import a1sini_au, a_phot_au, mass_function_au

a1sini = a1sini_au(K_draws, P_days, e)                               # AU
fm_spec = mass_function_au(a1sini, P_days)                           # (m2 sin i)^3 / M^2, Msun
aphot = a_phot_au(ti_chains["a_phot_mas"], ti_chains["plx_mas"])       # AU, beta = 0
fm_ast = mass_function_au(aphot, P_days)                             # m2^3 / M^2, Msun
print(f"a1 sin i = {np.median(a1sini):.4f} AU; fm_spec = {np.median(fm_spec):.3f} Msun")
print(f"a_phot   = {np.median(aphot):.4f} AU; fm_ast  = {np.median(fm_ast):.3f} Msun")
print(f"truth: sin i = {np.sin(truth.i_rad):.3f}, so fm_ast / fm_spec should be 1/sin^3 i = {np.sin(truth.i_rad)**-3:.2f}; "
      f"measured {np.median(fm_ast) / np.median(fm_spec):.2f}")

## The companion-mass solve, and its labels

`solve_companion_mass` inverts the mass function for m₂ given an **assumed** m₁
and sin i. The posterior adapters wrap it for draws: with `sin_i=1.0` you get the
minimum mass; with measured draws (`MeasuredSinI`, row-aligned with the mass
function) you get a true mass. Honesty note: a *fixed* `sin_i < 1` is an
assumption yet carries the same label as a measured inclination — read
`_meta["sin_i_mode"]` to tell them apart.

In [ ]:
from orblet.interpret.companion_mass import (solve_companion_mass, companion_mass_from_rv_posterior,
                                           companion_mass_from_astrometric_posterior)
from orblet.interpret.flux_ratio import MeasuredSinI

m2_min = solve_companion_mass(fm_spec, truth.m1_msun, 1.0)
print(f"minimum companion mass (sin i = 1): {np.median(m2_min):.2f} Msun")

rv_result = {"chains": {"fm_spec_msun": fm_spec}}
proj = companion_mass_from_rv_posterior(rv_result, primary_mass_prior=truth.m1_msun, sin_i=1.0)
sin_i_meas = MeasuredSinI(np.sin(ti_chains["inc_rad"]))                # measured, row-aligned
true = companion_mass_from_rv_posterior(rv_result, primary_mass_prior=("Normal", truth.m1_msun, 0.05),
                                        sin_i=sin_i_meas)
for label, res in (("sin i = 1", proj), ("measured sin i", true)):
    print(f"{label:>15}: m2 = {res['summary']['m2']['median']:.2f} Msun; "
          f"label = {res['_meta']['mass_convention']}, sin_i_mode = {res['_meta']['sin_i_mode']}")

# The astrometric adapter wants a Thiele-Innes observables chain: the A..G keys
# (or a basis sentinel in the chain's own _meta) plus the mass-function draws.
ti_result = {"chains": {**{k: ti_chains[k] for k in ("A_mas", "B_mas", "F_mas", "G_mas")},
                        "fm_ast_msun": fm_ast, "_meta": {"basis": "thiele_innes"}}}
ast = companion_mass_from_astrometric_posterior(ti_result, primary_mass_prior=truth.m1_msun)
print(f"astrometric route (beta = 0 assumed): m2 = {ast['summary']['m2']['median']:.2f} Msun; "
      f"label = {ast['_meta']['mass_convention']}")

## The astrometric upper limit and the bridges

When astrometry does not detect the orbit, `astrometric_upper_limit_au` turns
the photocentre draws into a credible bound and `companion_mass_bracket` into a
mass range; `consistency_verdict` checks it against the spectroscopic axis. The
bridge functions carry one channel's chain into the other's priors.

In [ ]:
from orblet.interpret.astrometric_upper_limit import (astrometric_upper_limit_au, companion_mass_bracket,
                                                    consistency_verdict)
from orblet import a1_sini_from_rv_chain, astrometric_priors_from_rv_chain, predicted_k_kms_from_astrometric_chain
from orblet.constants import OMEGA_CONVENTION_PRIMARY

a_up = astrometric_upper_limit_au(aphot, percentile=95.0)
lo, hi = companion_mass_bracket(float(np.median(a1sini)), a_up, truth.P_days, truth.m1_msun)
print(f"95 % upper bound on a_phot: {a_up:.4f} AU -> companion mass bracket {lo:.2f} .. {hi:.2f} Msun")
print("consistent with the spectroscopic axis:", consistency_verdict(a_up, float(np.percentile(a1sini, 16))))

rv_chain = {"P_days": P_days, "e": e, "K_kms": K_draws, "omega_rad": np.full(n, truth.omega_rad)}
print("a1 sin i from an RV chain (AU):", a1_sini_from_rv_chain(rv_chain, parallax_mas=truth.parallax_mas)["summary_au"])

# A hand-built chain must declare its omega convention (the prior bridge refuses to guess);
# the priors come back in the same primary frame the astrometric engine consumes.
priors = astrometric_priors_from_rv_chain({**rv_chain, "_meta": {"omega_convention": OMEGA_CONVENTION_PRIMARY}})
print("astrometric priors from the RV chain:", priors)
print(f"omega prior centre {priors['ω'][1]:.3f} rad vs truth {((truth.omega_rad + np.pi) % (2 * np.pi)) - np.pi:.3f} rad (primary frame, wrapped)")
k_pred = predicted_k_kms_from_astrometric_chain({"P_days": P_days, "e": e, "a_phot_au": aphot, "inc_rad": ti_chains["inc_rad"]})
print(f"K predicted from the astrometric chain (beta = 0): {k_pred['summary_kms']['median']:.2f} km/s vs measured {np.median(K_draws):.2f}")

## Is it really dark? Flux ratio and the AMRF

`signed_photocentre_axis_ratio` gives a_phot / a₁ for a mass ratio q and flux
ratio β; `beta_branches_from_axis_ratio` inverts it (two branches). The
astrometric mass-ratio function `amrf` and its boundaries classify a system
against what a main-sequence companion could produce. The shipped
main-sequence flux-ratio relation is a placeholder, so pass your own
`beta_relation(q, m1)` and validate it first.

In [ ]:
from orblet.interpret.flux_ratio import signed_photocentre_axis_ratio, beta_branches_from_axis_ratio
from orblet.interpret.amrf import (amrf, amrf_curve, ms_amrf_boundary, triple_amrf_boundary,
                                 classify_amrf, validate_beta_relation)

q = np.linspace(0.05, 1.0, 20)
print("a_phot / a1 at beta = 0.2:", np.round(signed_photocentre_axis_ratio(q, 0.2), 2)[:6], "...")
branches = beta_branches_from_axis_ratio(r=0.8, q=0.5)
print("beta branches for a_phot / a1 = 0.8 at q = 0.5:", {k: (round(float(v), 3) if np.isscalar(v) or np.ndim(v) == 0 else v) for k, v in branches.items()})

def my_beta(q, m1_msun):
    """ILLUSTRATIVE main-sequence flux ratio: L propto M^4, so beta = q^4 (q <= 1)."""
    return np.asarray(q, dtype=float) ** 4

validate_beta_relation(my_beta, truth.m1_msun)      # raises if the relation is unusable
A_obs = amrf(float(np.median(ti_chains["a_phot_mas"])), float(np.median(ti_chains["plx_mas"])), truth.P_days, truth.m1_msun)
print(f"AMRF of this system: {float(A_obs):.3f}; MS boundary {ms_amrf_boundary(truth.m1_msun, beta_relation=my_beta):.3f}; "
      f"unresolved-pair boundary {triple_amrf_boundary(truth.m1_msun, beta_relation=my_beta):.3f}")
verdict = classify_amrf(float(np.median(ti_chains["a_phot_mas"])), float(np.median(ti_chains["plx_mas"])), truth.P_days,
                        truth.m1_msun, beta_relation=my_beta)
print("label:", verdict["label"], "(a statement about which main-sequence constructions are excluded, not a probability)")

fig, ax = plt.subplots(figsize=(6, 3))
qq = np.linspace(0.01, 1.0, 200)
for beta in (0.0, 0.1, 0.3):
    ax.plot(qq, amrf_curve(qq, beta), label=f"beta = {beta}")
ax.plot(qq, amrf_curve(qq, my_beta(qq, truth.m1_msun)), "k--", label="main sequence (illustrative)")
ax.set_xlabel("mass ratio q"); ax.set_ylabel("AMRF"); ax.legend()
plt.show()

## Posterior summaries and the prior family

`chain_stats` summarises draws (with a circular summary for angles);
`ti_to_kepler` / `to_nss_convention` convert chains between conventions; the
prior classes are ordinary objects with `logpdf` and `sample`, usable in your
own sampler. Two defaults to know: the companion-mass ceiling is 5 M☉ and the
default total mass is 1 M☉ — override both for compact-object work.

In [ ]:
from orblet.chain_stats import chain_quantiles, chain_credible_interval, chain_circular_summary, chain_summary_table
from orblet import to_nss_convention
from orblet.priors import (UniformInFrequencyPeriodPrior, EccOmegaDiskPrior, CosUniformInclinationPrior,
                                   LogUniformPrior, TruncatedNormalPrior, NormalPrior,
                                   default_companion_priors, default_system_priors)

print("K quantiles:", chain_quantiles(K_draws))
print("K 95 % interval:", chain_credible_interval(K_draws, level=0.95))
print("omega (circular summary):", chain_circular_summary(ti_chains["omega_rad"]))
print(chain_summary_table({"K_kms": K_draws, "a_phot_mas": ti_chains["a_phot_mas"], "m2_min_msun": m2_min}))

nss = to_nss_convention({**ti_chains, "P_days": P_days, "e": e, "_meta": {"epoch_ref_mjd": EPOCH_REF}}, include_keplerian=False)
print("NSS-convention keys added:", sorted(set(nss) - set(ti_chains) - {"P_days", "e", "_meta"}))

rng = np.random.default_rng(0)
p_prior = UniformInFrequencyPeriodPrior(lo=10.0, hi=3000.0)
print("uniform-in-frequency prior favours SHORT periods: logpdf(100 d) - logpdf(1000 d) =",
      round(p_prior.logpdf(100.0) - p_prior.logpdf(1000.0), 2))
print("log-uniform:", LogUniformPrior(lo=10.0, hi=3000.0).logpdf(100.0), "; normal:", NormalPrior(mu=1.0, sigma=0.1).logpdf(1.05))
print("truncated normal sample:", np.round(TruncatedNormalPrior(mu=1.0, sigma=0.2, lower=0.5, upper=2.0).sample(rng, 3), 3))
print("(e, omega) disk prior logpdf at h = k = 0.3:", EccOmegaDiskPrior().logpdf(0.3, 0.3))
print("isotropic inclination prior sample (latent u):", np.round(CosUniformInclinationPrior().sample(rng, 3), 3))
print("engine defaults:", default_companion_priors(truth.P_days)["mass"], default_system_priors()["M"])

Assumptions this notebook made, in order of appearance: the primary's mass;
a dark companion for anything derived from the photocentre; an inclination when
one was fixed rather than measured; an illustrative flux-ratio relation for the
AMRF. Next: `05_qc_and_preparation.ipynb`.